# Sora / LFM — Pretrained QLoRA training

This notebook fine-tunes **Qwen2.5-1.5B-Instruct** with QLoRA. Enable a Kaggle GPU accelerator before running. It does not train from random initialization and does not overwrite the base model.

**Before training:** add a JSONL chat dataset as a Kaggle input, or upload it to `/kaggle/working/data/sora_chat.jsonl`. Each line should contain `{"messages":[{"role":"user","content":"..."},{"role":"assistant","content":"..."}]}`. Do not train on private conversations without permission.

In [ ]:
# 1. Verify GPU. Stop here if no CUDA GPU is available.
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('In Kaggle: Settings → Accelerator → GPU, then restart the session.')
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# 2. Install training dependencies
%pip install -q -U 'transformers>=4.46,<5' 'datasets>=3.0' 'peft>=0.13' 'accelerate>=1.0' 'bitsandbytes>=0.44' safetensors

In [ ]:
# 3. Get the latest repository branch with the training script
import os, subprocess, pathlib
repo = pathlib.Path('/kaggle/working/LFM-on-web-model')
if repo.exists():
    subprocess.run(['git', '-C', str(repo), 'fetch', 'origin', 'pretrained-qwen-lora'], check=True)
    subprocess.run(['git', '-C', str(repo), 'checkout', 'pretrained-qwen-lora'], check=True)
    subprocess.run(['git', '-C', str(repo), 'pull', '--ff-only', 'origin', 'pretrained-qwen-lora'], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', 'pretrained-qwen-lora', 'https://github.com/lstudiolab/LFM-on-web-model.git', str(repo)], check=True)
print('Training script:', repo / 'python/train_sora_lora.py')
print('Working directory:', os.getcwd())

In [ ]:
# 4. Locate the dataset. Set DATA_PATH to the actual JSONL path if your dataset is mounted elsewhere.
from pathlib import Path
candidates = [Path('/kaggle/working/data/sora_chat.jsonl')] + list(Path('/kaggle/input').glob('**/*.jsonl'))
found = [p for p in candidates if p.is_file()]
for p in found[:30]: print(p)
DATA_PATH = str(found[0]) if found else '/kaggle/working/data/sora_chat.jsonl'
print('Selected:', DATA_PATH)
if not Path(DATA_PATH).is_file():
    raise FileNotFoundError('No JSONL chat dataset found. Add a dataset in Kaggle → Add Input, or upload data/sora_chat.jsonl, then rerun this cell.')

In [ ]:
# 5. Short smoke test first. Output is written to Kaggle working storage.
import subprocess, sys
cmd = [sys.executable, str(repo / 'python/train_sora_lora.py'), '--data', DATA_PATH, '--output', '/kaggle/working/sora-qwen-lora-smoke', '--max-steps', '20', '--limit', '64', '--save-steps', '10', '--logging-steps', '1']
print('Running:', ' '.join(cmd))
subprocess.run(cmd, check=True)

## Longer training (run only after the smoke test succeeds)

Run the next cell for 500 steps. It creates a separate output folder. Save/download the resulting adapter before the Kaggle session ends; `/kaggle/working` is temporary. A successful smoke test does not guarantee that 500 steps will improve quality, so evaluate the adapter against held-out prompts.

In [ ]:
# 6. Optional longer run — execute manually after reviewing smoke-test output.
cmd = [sys.executable, str(repo / 'python/train_sora_lora.py'), '--data', DATA_PATH, '--output', '/kaggle/working/sora-qwen-lora', '--max-steps', '500', '--save-steps', '100', '--logging-steps', '10']
print('Running:', ' '.join(cmd))
subprocess.run(cmd, check=True)

In [ ]:
# 7. Check that an adapter was saved
from pathlib import Path
for p in [Path('/kaggle/working/sora-qwen-lora-smoke/adapter-final'), Path('/kaggle/working/sora-qwen-lora/adapter-final')]:
    if p.exists():
        print('Adapter files:', p)
        for f in sorted(p.iterdir()): print(f.name, f.stat().st_size, 'bytes')